# Garmin Connect — data exploration

Explore what the **garminconnect** Python library returns for each dataset used by the Garmin MCP coach tools (`mcp-servers/garmin-mcp/server.py`).

**MCP tools → library calls**

| MCP tool | `Garmin` client method(s) |
|---|---|
| `get_profile` | `get_user_profile()` |
| `get_activities` | `get_activities_by_date()` + `get_activity_hr_in_timezones()` |
| `get_weekly_stats` | same as activities, aggregated per calendar week |
| `get_race_predictions` | `get_race_predictions()` |
| `get_events` | `get_scheduled_workouts(year, month)` |
| `get_weekly_report` | bundle of the above |

**Credentials:** set `GARMIN_EMAIL` and `GARMIN_PASSWORD` in the repo-root `.env` file.

## Setup

In [ ]:
import json
import os
import sys
from datetime import date, timedelta
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "garmin-connect":
    REPO_ROOT = NOTEBOOK_DIR.parents[1]
else:
    REPO_ROOT = NOTEBOOK_DIR

GARMIN_MCP_DIR = REPO_ROOT / "mcp-servers" / "garmin-mcp"
sys.path.insert(0, str(GARMIN_MCP_DIR))

env_path = REPO_ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())

try:
    from garminconnect import Garmin
except ImportError:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "garminconnect"])
    from garminconnect import Garmin

import server as garmin_mcp


def preview(label: str, data, *, max_chars: int = 6000) -> None:
    """Pretty-print JSON-like payloads with a size cap."""
    print(f"\n=== {label} ===")
    text = json.dumps(data, indent=2, default=str)
    if len(text) > max_chars:
        print(text[:max_chars])
        print(f"\n... truncated ({len(text) - max_chars} more chars)")
    else:
        print(text)


GARMIN_EMAIL = os.environ.get("GARMIN_EMAIL")
GARMIN_PASSWORD = os.environ.get("GARMIN_PASSWORD")
print(f"Repo root: {REPO_ROOT}")
print(f"Garmin email set: {bool(GARMIN_EMAIL)}")

## Connect to Garmin Connect

In [ ]:
if not GARMIN_EMAIL or not GARMIN_PASSWORD:
    raise RuntimeError(
        "Set GARMIN_EMAIL and GARMIN_PASSWORD in the repo-root .env file."
    )

client = Garmin(email=GARMIN_EMAIL, password=GARMIN_PASSWORD)
client.login()
print("Logged in to Garmin Connect.")

## 1. User profile — `get_user_profile()`

MCP `get_profile` reads `userData` and extracts weight, height, birth date, VO2 max, lactate threshold, and training-day preferences.

In [ ]:
raw_profile = client.get_user_profile()
preview("Raw get_user_profile()", raw_profile)

user_data = (raw_profile or {}).get("userData", {})
preview("userData keys", sorted(user_data.keys()))

# Fields the MCP tool maps from userData
mcp_profile = {
    "weight": round(user_data.get("weight", 0) / 1000, 2),
    "height": user_data.get("height"),
    "birthDate": user_data.get("birthDate"),
    "vo2MaxRunning": user_data.get("vo2MaxRunning"),
    "lactateThresholdSpeed": round(
        (user_data.get("lactateThresholdSpeed", 0) * 1000 / 60), 2
    ),
    "lactateThresholdHeartRate": user_data.get("lactateThresholdHeartRate"),
    "availableTrainingDays": user_data.get("availableTrainingDays", []),
    "preferredLongTrainingDays": user_data.get("preferredLongTrainingDays", []),
}
preview("MCP get_profile() shape", mcp_profile)

## 2. Activities — `get_activities_by_date(start, end)`

MCP `get_activities` filters to `activityType.typeKey == "running"` and maps distance, duration, pace, HR, and training-effect fields.

In [ ]:
DAYS_BACK = 14
today = date.today()
start_date = (today - timedelta(days=DAYS_BACK)).isoformat()
end_date = today.isoformat()

raw_activities = client.get_activities_by_date(start_date, end_date)
print(f"Activities from {start_date} to {end_date}: {len(raw_activities)}")

if raw_activities:
    preview("First activity (full)", raw_activities[0])
    preview("First activity keys", sorted(raw_activities[0].keys()))

running_activities = [
    a
    for a in raw_activities
    if a.get("activityType", {}).get("typeKey") == "running"
]
print(f"Running activities: {len(running_activities)}")

if running_activities:
    a = running_activities[0]
    avg_speed = a.get("averageSpeed")
    max_speed = a.get("maxSpeed")
    mcp_row = {
        "name": a.get("activityName"),
        "type": a.get("activityType", {}).get("typeKey"),
        "distance_km": round((a.get("distance", 0) or 0) / 1000, 2),
        "duration_min": round((a.get("movingDuration", 0) or 0) / 60, 2),
        "avg_pace_min_per_km": round((1000 / avg_speed) / 60, 2) if avg_speed else None,
        "max_pace_min_per_km": round((1000 / max_speed) / 60, 2) if max_speed else None,
        "max_hr": a.get("maxHR"),
        "avg_hr": a.get("averageHR"),
        "aerobic_te": round(a.get("aerobicTrainingEffect", 0) or 0, 2),
        "anaerobic_te": round(a.get("anaerobicTrainingEffect", 0) or 0, 2),
        "training_effect": a.get("trainingEffectLabel"),
    }
    preview("MCP activity row (no HR zones yet)", mcp_row)

## 3. HR zones per activity — `get_activity_hr_in_timezones(activity_id)`

MCP `get_activities` (with `include_hr_zones=True`) adds zone 1–5 minutes from this endpoint.

In [ ]:
if not running_activities:
    print("No running activities in range — skip HR zone fetch.")
else:
    sample_activity = running_activities[0]
    activity_id = sample_activity.get("activityId")
    raw_zones = client.get_activity_hr_in_timezones(str(activity_id))
    preview(f"Raw HR zones for activity {activity_id}", raw_zones)

    from zones import normalize_hr_zones, zones_to_minute_columns

    zones = normalize_hr_zones(raw_zones)
    preview("Normalized zones (zone_number -> seconds)", zones)
    preview("MCP zone columns (minutes)", dict(zip(
        [f"zone_{i}_min" for i in range(1, 6)],
        zones_to_minute_columns(zones),
    )))

## 4. Race predictions — `get_race_predictions()`

MCP converts seconds to minutes (5K/10K) or hours (half / marathon).

In [ ]:
raw_predictions = client.get_race_predictions()
preview("Raw get_race_predictions()", raw_predictions)

mcp_predictions = {
    "5k_min": round((raw_predictions.get("time5K", 0) or 0) / 60, 2),
    "10k_min": round((raw_predictions.get("time10K", 0) or 0) / 60, 2),
    "half_marathon_h": round(
        (raw_predictions.get("timeHalfMarathon", 0) or 0) / 3600, 2
    ),
    "marathon_h": round((raw_predictions.get("timeMarathon", 0) or 0) / 3600, 2),
}
preview("MCP get_race_predictions() shape", mcp_predictions)

## 5. Calendar events — `get_scheduled_workouts(year, month)`

MCP `get_events` scans upcoming months, keeps `itemType == "event"`, and reads `completionTarget` for race goals.

In [ ]:
year, month = today.year, today.month
raw_calendar = client.get_scheduled_workouts(year=year, month=month)
preview(f"Raw get_scheduled_workouts({year}, {month})", raw_calendar)

calendar_items = raw_calendar.get("calendarItems", []) if isinstance(raw_calendar, dict) else []
print(f"calendarItems count: {len(calendar_items)}")

events = [item for item in calendar_items if item.get("itemType") == "event"]
preview(f"Events in {year}-{month:02d}", events)

if events:
    e = events[0]
    completion_target = e.get("completionTarget") or {}
    mcp_event_row = {
        "title": e.get("title"),
        "item_type": e.get("itemType"),
        "date": e.get("date"),
        "target_value": completion_target.get("value"),
        "target_unit": completion_target.get("unit"),
        "target_unit_type": completion_target.get("unitType"),
    }
    preview("MCP event row shape", mcp_event_row)

## 6. Weekly stats (derived)

Not a single API call — MCP aggregates running activities + HR zones into calendar-week rows (`zones.weekly_stats_rows`).

In [ ]:
from zones import normalize_hr_zones, weekly_stats_rows

WEEKS = 4
lookback_days = WEEKS * 7 - 1
ws_start = (today - timedelta(days=lookback_days)).isoformat()
ws_activities = client.get_activities_by_date(ws_start, end_date)
ws_running = [
    a
    for a in ws_activities
    if a.get("activityType", {}).get("typeKey") == "running"
]

activity_zones = {}
for activity in ws_running:
    aid = activity.get("activityId")
    if aid is None:
        continue
    raw = client.get_activity_hr_in_timezones(str(aid))
    activity_zones[aid] = normalize_hr_zones(raw)

headers = garmin_mcp.WEEKLY_STATS_HEADERS
rows = weekly_stats_rows(ws_running, activity_zones, today, num_blocks=WEEKS)
preview("MCP weekly stats table", {"Headers": headers, "Rows": rows})

## 7. MCP tool wrappers (optional)

Call the same functions the MCP server exposes, using the live client patched in.

In [ ]:
from unittest.mock import patch

with patch.object(garmin_mcp, "_get_client_or_error", return_value=(client, None)):
    preview("get_profile()", garmin_mcp.get_profile())
    preview("get_race_predictions()", garmin_mcp.get_race_predictions())
    preview("get_events(months_ahead=3)", garmin_mcp.get_events(months_ahead=3))
    preview(
        "get_activities(days_back=7, include_hr_zones=False)",
        garmin_mcp.get_activities(days_back=7, include_hr_zones=False),
    )
    preview("get_weekly_stats(weeks=4)", garmin_mcp.get_weekly_stats(weeks=4))